In [2]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

In [3]:
GK_DATA = Path("GK_Data")

# 1. Load labels
dataset = pd.read_csv(GK_DATA / "gk_dataset_final.csv")
print(f"Total: {len(dataset)} keepers")
print(dataset["status"].value_counts())

# 2. Load definition files
with open(GK_DATA / "player_kpi_definitions.json") as f:
    kpi_defs = {d["id"]: d["name"] for d in json.load(f).get("data", [])}

with open(GK_DATA / "player_score_definitions.json") as f:
    score_defs = {d["id"]: d["name"] for d in json.load(f).get("data", [])}


# 3. Load KPIs for a single keeper
def load_keeper_match_kpis(player_id, match_dirs_str):
    """Load player_kpis for a keeper from all his matches."""
    all_kpis = []
    if not isinstance(match_dirs_str, str) or not match_dirs_str:
        return all_kpis

    for match_dir in match_dirs_str.split("|"):
        match_dir = match_dir.strip()
        if not match_dir:
            continue
        # Search in all competition-season folders
        for cs_dir in (GK_DATA / "competitions").iterdir():
            if not cs_dir.is_dir():
                continue
            pkpi_path = cs_dir / match_dir / "player_kpis.json"
            if pkpi_path.exists():
                with open(pkpi_path) as f:
                    data = json.load(f).get("data", {})
                for side in ["squadHome", "squadAway"]:
                    for player in data.get(side, {}).get("players", []):
                        if (
                            player["id"] == player_id
                            and player.get("position") == "GOALKEEPER"
                            and player.get("matchShare", 0) >= 0.5
                        ):
                            kpis = {
                                k["kpiId"]: k["value"] for k in player.get("kpis", [])
                            }
                            kpis["matchShare"] = player["matchShare"]
                            kpis["playDuration"] = player.get("playDuration", 0)
                            all_kpis.append(kpis)
                break  # found, move to next match
    return all_kpis

Total: 693 keepers
status
STAYED     435
DROPPED    128
PLAYS       99
BENCH       31
Name: count, dtype: int64


In [4]:
print(dataset.columns.tolist())

['playerId', 'name', 'age', 'birthdate', 'status', 'origin_team', 'origin_comp', 'origin_season', 'origin_median', 'origin_matches', 'current_team', 'current_comp', 'current_season', 'current_median', 'current_matches', 'step', 'direction', 'origin_match_dirs', 'current_match_dirs']


In [5]:
# 4. Example: load data for first keeper
row = dataset.iloc[4]
kpis = load_keeper_match_kpis(row["playerId"], row["origin_match_dirs"])
print(
    f"{row['name']}: {len(kpis)} matches, {len(kpis[0]) if kpis else 0} KPIs per match"
)

Simon Simoni: 13 matches, 280 KPIs per match


In [6]:
all_rows = []

for _, row in dataset.iterrows():
    match_kpis = load_keeper_match_kpis(row["playerId"], row["origin_match_dirs"])
    
if match_kpis:
    player_entry = row.to_dict() 
    player_entry.pop('origin_match_dirs', None)
    player_entry.pop('current_match_dirs', None)
    kpi_ids = {k for m in match_kpis for k in m.keys() if isinstance(k, int)}
    for kpi_id in kpi_ids:
        values = [m[kpi_id] for m in match_kpis if kpi_id in m]
        kpi_name = kpi_defs.get(kpi_id, f"KPI_{kpi_id}")
        player_entry[f"mean_{kpi_name}"] = np.mean(values)
            
        all_rows.append(player_entry)
df_final = pd.DataFrame(all_rows)

print(f"Final dataset ready with {df_final.shape[0]} keepers and {df_final.shape[1]} features.")
print(df_final.columns.tolist())

Final dataset ready with 527 keepers and 544 features.
['playerId', 'name', 'age', 'birthdate', 'status', 'origin_team', 'origin_comp', 'origin_season', 'origin_median', 'origin_matches', 'current_team', 'current_comp', 'current_season', 'current_median', 'current_matches', 'step', 'direction', 'mean_BYPASSED_OPPONENTS', 'mean_BYPASSED_OPPONENTS_NUMBER', 'mean_BYPASSED_DEFENDERS', 'mean_BYPASSED_OPPONENTS_WO_VERIFICATION', 'mean_BYPASSED_DEFENDERS_WO_VERIFICATION', 'mean_BYPASSED_OPPONENTS_BY_ACTION_SECOND_BALL', 'mean_BYPASSED_OPPONENTS_RECEIVING', 'mean_BYPASSED_OPPONENTS_RECEIVING_NUMBER', 'mean_BYPASSED_OPPONENTS_RECEIVING_DUETO_VERIFICATION', 'mean_NEUTRAL_PLAY_NUMBER', 'mean_REVERSE_PLAY_ADDED_OPPONENTS', 'mean_REVERSE_PLAY_NUMBER', 'mean_PLAYDURATION', 'mean_MATCH_SHARE', 'mean_BALL_LOSS_ADDED_OPPONENTS', 'mean_BALL_LOSS_REMOVED_TEAMMATES', 'mean_BALL_LOSS_NUMBER', 'mean_BALL_WIN_ADDED_TEAMMATES', 'mean_BALL_WIN_REMOVED_OPPONENTS', 'mean_BALL_WIN_NUMBER', 'mean_BYPASSED_OPPONENT

In [31]:
df_final.sample(100)

,playerId,name,age,birthdate,status,origin_team,origin_comp,origin_season,origin_median,origin_matches,...,mean_NEUTRAL_PASSES_TO_LANE_RIGHT_HALF_SPACE,mean_NEUTRAL_PASSES_TO_LANE_CENTER,mean_NEUTRAL_PASSES_TO_LANE_LEFT_HALF_SPACE,mean_NEUTRAL_PASSES_TO_LANE_LEFT_WING,mean_NEUTRAL_PASSES_AT_PHASE_IN_POSSESSION,mean_NEUTRAL_PASSES_AT_PHASE_ATTACKING_TRANSITION,mean_NEUTRAL_PASSES_AT_PHASE_SET_PIECE,mean_EXPECTED_SHOT_ASSISTS,mean_EXPECTED_GOAL_ASSISTS,mean_EXPECTED_PASSES
287,37461,Denis Rusu,35.0,1990-08-02,STAYED,AFC Unirea 04 Slobozia,superliga_romania,2024-2025,0.456571,40,...,1.4,5.361111,1.5,1.0,2.363636,1.3125,4.088235,0.041034,0.003584,14.59212
381,37461,Denis Rusu,35.0,1990-08-02,STAYED,AFC Unirea 04 Slobozia,superliga_romania,2024-2025,0.456571,40,...,1.4,5.361111,1.5,1.0,2.363636,1.3125,4.088235,0.041034,0.003584,14.59212
358,37461,Denis Rusu,35.0,1990-08-02,STAYED,AFC Unirea 04 Slobozia,superliga_romania,2024-2025,0.456571,40,...,1.4,5.361111,1.5,1.0,2.363636,1.3125,4.088235,0.041034,0.003584,14.59212
359,37461,Denis Rusu,35.0,1990-08-02,STAYED,AFC Unirea 04 Slobozia,superliga_romania,2024-2025,0.456571,40,...,1.4,5.361111,1.5,1.0,2.363636,1.3125,4.088235,0.041034,0.003584,14.59212
218,37461,Denis Rusu,35.0,1990-08-02,STAYED,AFC Unirea 04 Slobozia,superliga_romania,2024-2025,0.456571,40,...,1.4,5.361111,1.5,1.0,2.363636,1.3125,4.088235,0.041034,0.003584,14.59212
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,37461,Denis Rusu,35.0,1990-08-02,STAYED,AFC Unirea 04 Slobozia,superliga_romania,2024-2025,0.456571,40,...,1.4,5.361111,1.5,1.0,2.363636,1.3125,4.088235,0.041034,0.003584,14.59212
329,37461,Denis Rusu,35.0,1990-08-02,STAYED,AFC Unirea 04 Slobozia,superliga_romania,2024-2025,0.456571,40,...,1.4,5.361111,1.5,1.0,2.363636,1.3125,4.088235,0.041034,0.003584,14.59212
56,37461,Denis Rusu,35.0,1990-08-02,STAYED,AFC Unirea 04 Slobozia,superliga_romania,2024-2025,0.456571,40,...,1.4,5.361111,1.5,1.0,2.363636,1.3125,4.088235,0.041034,0.003584,14.59212
403,37461,Denis Rusu,35.0,1990-08-02,STAYED,AFC Unirea 04 Slobozia,superliga_romania,2024-2025,0.456571,40,...,1.4,5.361111,1.5,1.0,2.363636,1.3125,4.088235,0.041034,0.003584,14.59212
